# PHS564 — Lecture 14 (Student)
## Robustness and sensitivity: unmeasured confounding, negative controls, and pragmatic checks

### Acronyms used in this notebook
- **CI**: confidence interval
- **OR**: odds ratio
- **RR**: risk ratio

### Learning goals
- Quantify how strong unmeasured confounding must be to explain away an effect (at least one method):
  **E-value on the risk ratio (RR) scale** (OR requires an approximation; label clearly) or a simple bias-factor sensitivity.
- Use **negative controls** conceptually (negative control exposure/outcome) to detect residual confounding.
- Make robustness reporting standard: alternate specs, missingness summary, overlap diagnostics, and a short limitations statement.

### Required reading
- Hernán & Robins, [What If](https://miguelhernan.org/whatifbook): limitations/assumptions and how violations threaten causal interpretation (note: E-values are an external add-on).

**Rules for this notebook**
- Only edit cells marked **TODO**.
- Do not change the overall structure/cell order.


In [ ]:
# Colab bootstrap (run this first if you opened from a Colab badge)
# - Clones the repo into /content/PHS564 (if needed)
# - Installs requirements
# - Adds repo to sys.path

from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path


def _in_colab() -> bool:
    return "google.colab" in sys.modules


if _in_colab():
    REPO_URL = "https://github.com/vafaei-ar/PHS564.git"
    TARGET_DIR = Path("/content/PHS564")

    if not (TARGET_DIR / "requirements.txt").exists():
        print("Cloning course repo into Colab runtime...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(TARGET_DIR)], check=True)

    os.chdir(TARGET_DIR)

    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"], check=True)

    if str(TARGET_DIR) not in sys.path:
        sys.path.insert(0, str(TARGET_DIR))

    print("✓ Colab setup complete. Now run the rest of the notebook.")
else:
    print("Not running in Colab; skipping Colab bootstrap.")

### Scientific notes (read once)

- **E-values** are defined for **risk ratios**. If you have an odds ratio (OR), using the RR E-value is an approximation and should be labeled as such.
- The "tipping-point" grid below is a **toy calculation** for intuition; it is **not** a formal sensitivity analysis and should not be presented as a definitive bound.
- For publication-grade sensitivity analyses, consult the E-value literature and/or bias formulas tailored to your estimand and outcome model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

def e_value_rr(rr: float) -> float:
    """E-value for a risk ratio estimate RR (RR>=1). If RR<1, use 1/RR."""
    rr = float(rr)
    rr = rr if rr >= 1 else 1/rr
    return rr + sqrt(rr*(rr-1))

def e_value_rr_ci(rr: float, lo: float, hi: float):
    """E-value for estimate and lower CI bound (conservative)."""
    est = e_value_rr(rr)
    # Conservative: use CI bound closest to 1 on RR scale
    lo2, hi2 = float(lo), float(hi)
    if rr >= 1:
        bound = lo2
    else:
        bound = hi2
    bound_ev = e_value_rr(bound)
    return est, bound_ev


## 1) Plug in your project estimate
Enter your primary effect estimate as an RR (or convert if needed).

In [ ]:
rr_hat = 0.80   # TODO: your RR estimate
rr_lo  = 0.70   # TODO: lower 95% CI
rr_hi  = 0.92   # TODO: upper 95% CI

ev_est, ev_bound = e_value_rr_ci(rr_hat, rr_lo, rr_hi)
print('E-value (estimate):', round(ev_est, 3))
print('E-value (conservative CI bound):', round(ev_bound, 3))

## 2) Tipping-point grid (bias factor)
This is a **toy** sensitivity: assume an unmeasured binary confounder U with
- RR_UY: association of U with outcome
- RR_AU: association of treatment with U

Bias factor ≈ RR_AU × RR_UY (very simplified). We ask: how large must it be to move your RR to 1?

In [ ]:
rr_obs = rr_hat
rr_target = 1.0
needed_bias = (rr_target/rr_obs) if rr_obs < 1 else (rr_obs/rr_target)
print('Approx bias factor needed to move estimate to null:', round(needed_bias, 3))

RR_AU = np.linspace(1, 3, 50)
RR_UY = np.linspace(1, 3, 50)
B = np.outer(RR_AU, RR_UY)

plt.figure(figsize=(7,5))
plt.imshow(B, origin='lower', aspect='auto')
plt.colorbar(label='Bias factor (toy)')
plt.xticks([0, 24, 49], [1, 2, 3])
plt.yticks([0, 24, 49], [1, 2, 3])
plt.xlabel('RR_UY')
plt.ylabel('RR_AU')
plt.title('Toy tipping-point grid (bigger = more bias)')
plt.show()

print('Interpretation: you would need a combined bias factor ≈', round(needed_bias,3))

## 3) Negative controls worksheet
Write 2–3 candidates and why they might work.

**Negative control outcome (NCO):** affected by similar confounding, but not by treatment.
**Negative control exposure (NCE):** affects treatment assignment/confounding, but not the outcome.


- **NCO idea:** TODO
- **Why:** TODO

- **NCE idea:** TODO
- **Why:** TODO


## 4) Robustness reporting checklist (copy into report appendix)
- Primary estimand and effect measure (explicit)
- Overlap/positivity diagnostics
- Weight distribution + truncation (if used)
- One alternate specification
- One sensitivity analysis (E-value or bias-factor)
- Limitations that would change conclusions
